# Debug: Entrenamiento y Exportación YOLOv8
## Álvaro Zarabanda - 20251595006

Este notebook permite ejecutar paso a paso el entrenamiento y exportación del modelo YOLO para identificar exactamente dónde ocurren los errores.

## 1. Importar Librerías y Configuración

In [4]:
from ultralytics import YOLO
import torch
from pathlib import Path
import os
import sys
import traceback

# Configuración
DATA_YAML = "dataset_synthetic/data.yaml"
BASE_MODEL = "yolov8n.pt"
EPOCHS = 1
BATCH_SIZE = 16
IMG_SIZE = 640
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
PROJECT_NAME = "runs/detect"
EXPERIMENT_NAME = "inventario"




print(" Librerías importadas correctamente")
print(f" Python: {sys.version}")
print(f" PyTorch: {torch.__version__}")
print(f" Ultralytics: {YOLO.__module__}")
print(f"  Device: {'CUDA' if torch.cuda.is_available() else 'CPU'}")

 Librerías importadas correctamente
 Python: 3.13.9 (main, Oct 14 2025, 00:00:00) [GCC 15.2.1 20250808 (Red Hat 15.2.1-1)]
 PyTorch: 2.9.1+cu128
 Ultralytics: ultralytics.models.yolo.model
  Device: CPU


In [5]:
# Información del sistema
print(f"📊 Configuración:")
print(f"   - Dataset: {DATA_YAML}")
print(f"   - Modelo base: {BASE_MODEL}")
print(f"   - Épocas: {EPOCHS}")
print(f"   - Batch size: {BATCH_SIZE}")
print(f"   - Tamaño imagen: {IMG_SIZE}x{IMG_SIZE}")
print(f"   - Dispositivo: {DEVICE.upper()}")
print()

if DEVICE == 'cuda':
    print(f"🎮 GPU detectada: {torch.cuda.get_device_name(0)}")
    print(f"   Memoria disponible: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    print()
else:
    print("⚠️  Entrenando en CPU (será más lento)")
    print()

📊 Configuración:
   - Dataset: dataset_synthetic/data.yaml
   - Modelo base: yolov8n.pt
   - Épocas: 1
   - Batch size: 16
   - Tamaño imagen: 640x640
   - Dispositivo: CPU

⚠️  Entrenando en CPU (será más lento)



## 2. Verificar Dataset

In [6]:
# Verificar existencia de archivos
dataset_path = Path(DATA_YAML)

if dataset_path.exists():
    print(f" Dataset YAML encontrado: {dataset_path}")
    
    # Leer contenido
    with open(dataset_path, 'r') as f:
        content = f.read()
    print("\n Contenido del data.yaml:")
    print(content)
    
    # Verificar imágenes
    train_dir = Path("dataset/images/train")
    val_dir = Path("dataset/images/val")
    
    train_imgs = len(list(train_dir.glob("*.jpg"))) if train_dir.exists() else 0
    val_imgs = len(list(val_dir.glob("*.jpg"))) if val_dir.exists() else 0
    
    print(f"\n Estadísticas:")
    print(f"   - Imágenes train: {train_imgs}")
    print(f"   - Imágenes val: {val_imgs}")
else:
    print(f" Dataset YAML NO encontrado: {dataset_path}")

 Dataset YAML encontrado: dataset_synthetic/data.yaml

 Contenido del data.yaml:
# Dataset Sintético - Inventario Salón de Cómputo
# Generado automáticamente con múltiples objetos por imagen

path: /run/media/SoporteOATI/HDD/Maestria/Repositorios/BigData-CNN/inventario/dataset_synthetic
train: images/train
val: images/val

# Classes
nc: 6
names: ['cpu', 'mesa', 'mouse', 'pantalla', 'silla', 'teclado']


 Estadísticas:
   - Imágenes train: 1238
   - Imágenes val: 312


## 3. Cargar Modelo YOLOv8n

In [7]:
try:
    print(f" Cargando modelo: {BASE_MODEL}")
    model = YOLO(BASE_MODEL)
    print(" Modelo cargado exitosamente")
    print(f" Resumen del modelo:")
    print(f"   - Nombre: {model.model_name}")
    print(f"   - Tarea: {model.task}")
except Exception as e:
    print(f" Error al cargar modelo: {e}")
    traceback.print_exc()

 Cargando modelo: yolov8n.pt
 Modelo cargado exitosamente
 Resumen del modelo:
   - Nombre: yolov8n.pt
   - Tarea: detect


## 4. Entrenar Modelo

In [8]:
try:
    print(" Iniciando entrenamiento...")
    print(f"   - Épocas: {EPOCHS}")
    print(f"   - Batch size: {BATCH_SIZE}")
    print(f"   - Tamaño imagen: {IMG_SIZE}")
    print(f"   - Dispositivo: {DEVICE}")
    
    results = model.train(
        data=DATA_YAML,
        epochs=EPOCHS,
        batch=BATCH_SIZE,
        imgsz=IMG_SIZE,
        device=DEVICE,
        project=PROJECT_NAME,
        name=EXPERIMENT_NAME,
        
        # Optimizaciones
        optimizer='AdamW',
        lr0=0.01,           # Learning rate inicial
        lrf=0.01,           # Learning rate final
        momentum=0.937,
        weight_decay=0.0005,
        warmup_epochs=3.0,
        warmup_momentum=0.8,
        warmup_bias_lr=0.1,
        
        # Data augmentation (importante para dataset sintético)
        hsv_h=0.015,        # Hue
        hsv_s=0.7,          # Saturation
        hsv_v=0.4,          # Value
        degrees=10.0,       # Rotación
        translate=0.1,      # Translación
        scale=0.5,          # Escala
        shear=0.0,          # Shear
        perspective=0.0,    # Perspective
        flipud=0.0,         # Flip vertical
        fliplr=0.5,         # Flip horizontal
        mosaic=1.0,         # Mosaic augmentation
        mixup=0.0,          # Mixup augmentation
        copy_paste=0.0,     # Copy-paste augmentation (ya lo hicimos manualmente)
        
        # Otras configuraciones
        patience=20,        # Early stopping
        save=True,
        save_period=10,     # Guardar cada 10 épocas
        cache=False,        # No usar cache (ahorra RAM)
        workers=8,          # Número de workers para carga de datos
        pretrained=True,    # Usar pesos pre-entrenados
        verbose=True,
        seed=42,            # Reproducibilidad
        deterministic=True,
        single_cls=False,   # Multi-clase
        rect=False,         # Rectangular training
        cos_lr=True,        # Cosine LR scheduler
        close_mosaic=10,    # Desactivar mosaic en últimas épocas
        amp=True,           # Automatic Mixed Precision (más rápido en GPU)
        fraction=1.0,       # Usar 100% del dataset
        profile=False,
        freeze=None,        # No congelar capas
        
        # Callbacks y logging
        plots=True,         # Generar plots
        save_json=False,
        save_hybrid=False,
        val=True,
        split='val',
    )
    
    
    print("\n Entrenamiento completado!")
    print(f" Resultados guardados en: {results.save_dir}")
    
except Exception as e:
    print(f"\n Error durante el entrenamiento: {e}")
    traceback.print_exc()

 Iniciando entrenamiento...
   - Épocas: 1
   - Batch size: 16
   - Tamaño imagen: 640
   - Dispositivo: cpu
WARNING ⚠️ 'save_hybrid' is deprecated and will be removed in the future.
Ultralytics 8.3.228 🚀 Python-3.13.9 torch-2.9.1+cu128 CPU (Intel Core i7-14700)
Ultralytics 8.3.228 🚀 Python-3.13.9 torch-2.9.1+cu128 CPU (Intel Core i7-14700)


engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=dataset_synthetic/data.yaml, degrees=10.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=1, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=inventario, nbs=64, nms=False, opset=None, optimize=False, optimizer=AdamW, overlap_mask=True, patience=20, perspective=0.0, plots=True, pose=12.0, pretrained=True, profile=False, project=runs/detect, rect=False, resume=Fals

/run/media/SoporteOATI/HDD/Maestria/Repositorios/BigData-CNN/.venv/lib64/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 6036.1±2400.4 MB/s, size: 112.8 KB)
val: Scanning /run/media/SoporteOATI/HDD/Maestria/Repositorios/BigData-CNN/inventario/dataset_synthetic/labels/val.cache... 750 images, 49 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 750/750 3.8Mit/s 0.0s0s
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 6036.1±2400.4 MB/s, size: 112.8 KB)
val: Scanning /run/media/SoporteOATI/HDD/Maestria/Repositorios/BigData-CNN/inventario/dataset_synthetic/labels/val.cache... 750 images, 49 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 750/750 3.8Mit/s 0.0s0s
Plotting labels to /run/media/SoporteOATI/HDD/Maestria/Repositorios/BigData-CNN/inventario/runs/detect/inventario/labels.jpg... 
Plotting labels to /run/media/SoporteOATI/HDD/Maestria/Repositorios/BigData

## 5. Validar Modelo Entrenado

In [13]:
try:
    print(" Validando modelo...")
    metrics = model.val()
    
    print("\n MÉTRICAS FINALES:")
    print(f"   - mAP50: {metrics.box.map50:.4f}")
    print(f"   - mAP50-95: {metrics.box.map:.4f}")
    print(f"   - Precisión: {metrics.box.mp:.4f}")
    print(f"   - Recall: {metrics.box.mr:.4f}")
    
    print("\n Validación completada")
    
except Exception as e:
    print(f"\n Error durante la validación: {e}")
    traceback.print_exc()

 Validando modelo...
Ultralytics 8.3.228 🚀 Python-3.13.9 torch-2.9.1+cu128 CPU (Intel Core i7-14700)
Model summary (fused): 72 layers, 3,006,818 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 5759.6±968.5 MB/s, size: 111.8 KB)
val: Scanning /run/media/SoporteOATI/HDD/Maestria/Repositorios/BigData-CNN/inventario/dataset/labels/val.cache... 312 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 312/312 1.5Mit/s 0.0s0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 20/20 2.4it/s 8.4s0.4s
                   all        312        312      0.928      0.817      0.913      0.799
                   cpu         42         42       0.94      0.746      0.865       0.79
                  mesa         56         56       0.92      0.911      0.981       0.88
                 mouse         53         53      0.929      0.742      0.834      0.697
              pantalla         51         51      0.9

## 6. Verificar Modelo Guardado

In [14]:
try:
    # Buscar el modelo guardado
    model_path = Path("runs/detect/inventario/weights/best.pt")
    
    if model_path.exists():
        size_mb = model_path.stat().st_size / (1024 * 1024)
        print(f" Modelo encontrado: {model_path}")
        print(f" Tamaño: {size_mb:.2f} MB")
        
        # Listar todos los archivos en weights
        weights_dir = model_path.parent
        print(f"\n Archivos en {weights_dir}:")
        for f in weights_dir.iterdir():
            if f.is_file():
                size = f.stat().st_size / (1024 * 1024)
                print(f"   - {f.name}: {size:.2f} MB")
    else:
        print(f" Modelo NO encontrado en: {model_path}")
        
except Exception as e:
    print(f" Error verificando modelo: {e}")
    traceback.print_exc()

 Modelo encontrado: runs/detect/inventario/weights/best.pt
 Tamaño: 5.96 MB

 Archivos en runs/detect/inventario/weights:
   - last.pt: 5.96 MB
   - best.pt: 5.96 MB
   - best.onnx: 11.70 MB


## 7. Exportar a ONNX

In [ ]:
import shutil

try:
    print("📦 Exportando modelo a ONNX...")
    
    # Cargar el mejor modelo
    best_model = YOLO("runs/detect/inventario/weights/best.pt")
    
    # Exportar a ONNX (se crea en la carpeta del modelo)
    onnx_path = best_model.export(
        format='onnx',
        imgsz=640,
        simplify=True,
        opset=12  # Versión de ONNX compatible
    )
    
    print(f" Modelo ONNX exportado: {onnx_path}")
    
    # Definir ruta de destino personalizada
    destino = Path("./models/inventario.onnx")
    destino.parent.mkdir(parents=True, exist_ok=True)  # Crear directorio si no existe
    
    # Copiar a la ruta deseada
    shutil.copy2(onnx_path, destino)
    print(f" Modelo copiado a: {destino}")
    
    # Verificar tamaños
    if Path(onnx_path).exists():
        size_mb = Path(onnx_path).stat().st_size / (1024 * 1024)
        print(f"   Tamaño original: {size_mb:.2f} MB")
    
    if destino.exists():
        size_mb = destino.stat().st_size / (1024 * 1024)
        print(f"   Tamaño destino: {size_mb:.2f} MB")
    
except Exception as e:
    print(f"\n Error durante la exportación a ONNX:")
    print(f"   Tipo de error: {type(e).__name__}")
    print(f"   Mensaje: {str(e)}")
    print("\n Stack trace completo:")
    traceback.print_exc()

 Exportando modelo a ONNX...
Ultralytics 8.3.228 🚀 Python-3.13.9 torch-2.9.1+cu128 CPU (Intel Core i7-14700)
Model summary (fused): 72 layers, 3,006,818 parameters, 0 gradients, 8.1 GFLOPs

PyTorch: starting from 'runs/detect/inventario_synthetic2/weights/epoch100.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 10, 8400) (17.6 MB)

ONNX: starting export with onnx 1.19.1 opset 12...
ONNX: slimming with onnxslim 0.1.74...
ONNX: export success ✅ 0.9s, saved as 'runs/detect/inventario_synthetic2/weights/epoch100.onnx' (11.7 MB)

Export complete (1.0s)
Results saved to /run/media/SoporteOATI/HDD/Maestria/Repositorios/BigData-CNN/inventario/runs/detect/inventario_synthetic2/weights
Predict:         yolo predict task=detect model=runs/detect/inventario_synthetic2/weights/epoch100.onnx imgsz=640  
Validate:        yolo val task=detect model=runs/detect/inventario_synthetic2/weights/epoch100.onnx imgsz=640 data=dataset_synthetic/data.yaml  
Visualize:       https://netron.app

## 8. Resumen y Verificación Final

In [16]:
import glob

print("="*60)
print("  RESUMEN FINAL")
print("="*60)

# Buscar todos los modelos generados
pt_models = glob.glob("runs/detect/*/weights/*.pt")
onnx_models = glob.glob("runs/detect/*/weights/*.onnx")

print("\n Modelos PyTorch (.pt):")
if pt_models:
    for model in pt_models:
        size_mb = Path(model).stat().st_size / (1024 * 1024)
        print(f"    {model} ({size_mb:.2f} MB)")
else:
    print("    No se encontraron modelos .pt")

print("\n Modelos ONNX (.onnx):")
if onnx_models:
    for model in onnx_models:
        size_mb = Path(model).stat().st_size / (1024 * 1024)
        print(f"    {model} ({size_mb:.2f} MB)")
else:
    print("    No se encontraron modelos .onnx")



  RESUMEN FINAL

 Modelos PyTorch (.pt):
    runs/detect/inventario/weights/last.pt (5.96 MB)
    runs/detect/inventario/weights/best.pt (5.96 MB)

 Modelos ONNX (.onnx):
    runs/detect/inventario/weights/best.onnx (11.70 MB)
